# Lab01S02 — Consulta GraphQL aos 1.000 repositórios mais populares do GitHub

| RQ | Pergunta | Métrica |
|:--:|----------|--------|
| 01 | Sistemas populares são maduros/antigos? | Idade do repositório |
| 02 | Recebem muita contribuição externa? | Total de PRs aceitas |
| 03 | Lançam releases com frequência? | Total de releases |
| 04 | São atualizados com frequência? | Tempo até última atualização |
| 05 | São escritos nas linguagens mais populares? | Linguagem primária |
| 06 | Possuem alto percentual de issues fechadas? | Razão issues fechadas / total |
| 07 | Sistemas em linguagens populares recebem mais contribuição, releases e updates? | PRs, releases e updates por grupo de linguagem |

## 1. Setup do Dataframe

In [1]:
import pandas as pd
import glob
from datetime import datetime
import os

In [4]:
# Busca o arquivo mais recente pelo timestamp no nome dentro da pasta data/
json_files = sorted(glob.glob("data/repos_lab01_s02*.json"), reverse=True)
csv_files = sorted(glob.glob("data/repos_lab01_s02*.csv"), reverse=True)

if json_files:
    latest = json_files[0]
    df = pd.read_json(latest, orient="records", convert_dates=["criado_em", "atualizado_em"])
    print(f"Carregado (JSON): {latest}")
elif csv_files:
    latest = csv_files[0]
    df = pd.read_csv(latest, parse_dates=["criado_em", "atualizado_em"])
    print(f"Carregado (CSV): {latest}")
else:
    raise FileNotFoundError("Nenhum arquivo repos_lab01_s02* encontrado em data/!")

print(f"Shape: {df.shape[0]} linhas x {df.shape[1]} colunas")

Carregado (JSON): data\repos_lab01_s0220260306_175638.json
Shape: 1000 linhas x 12 colunas


## 2. Visão geral dos dados

In [5]:
df.describe()

,estrelas,idade_dias,dias_desde_update,prs_aceitas,releases,total_issues,issues_fechadas,razao_fechadas
count,1000.000000,1000.000000,1000.00000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,58047.523000,2987.977000,0.02000,3954.169000,120.499000,5003.913000,4357.958000,0.772372
std,46453.366582,1494.144994,0.14007,9869.038634,199.321939,11879.957883,10703.980844,0.257072
min,29594.000000,33.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,34344.000000,1829.000000,0.00000,173.500000,0.000000,396.500000,249.500000,0.678175
50%,42956.000000,3061.000000,0.00000,743.000000,40.000000,1740.000000,1399.500000,0.867600
75%,62580.500000,4167.500000,0.00000,3197.750000,143.500000,5267.000000,4533.500000,0.958350
max,472883.000000,6538.000000,1.00000,94722.000000,1000.000000,232359.000000,218871.000000,1.000000


## 3. Estatísticas por RQ (medianas)

In [6]:
medians = pd.Series({
    "RQ01 - Idade (dias)":               df["idade_dias"].median(),
    "RQ02 - PRs aceitas":                df["prs_aceitas"].median(),
    "RQ03 - Releases":                   df["releases"].median(),
    "RQ04 - Dias desde último update":   df["dias_desde_update"].median(),
    "RQ06 - Razão issues fechadas":      df["razao_fechadas"].median(),
}, name="Mediana")

medians.to_frame()

,Mediana
RQ01 - Idade (dias),3061.0000
RQ02 - PRs aceitas,743.0000
RQ03 - Releases,40.0000
RQ04 - Dias desde último update,0.0000
RQ06 - Razão issues fechadas,0.8676


## 4. RQ05 — Sistemas populares são escritos nas linguagens mais populares?

Linguagens mais populares segundo o [GitHub Octoverse 2025](https://github.blog/news-insights/octoverse/octoverse-a-new-developer-joins-github-every-second-as-ai-leads-typescript-to-1/):
1. **TypeScript**
2. **Python**
3. **JavaScript**

In [7]:
LINGUAGENS_POPULARES = ["TypeScript", "Python", "JavaScript"]

lang_counts = df["linguagem"].fillna("N/A").value_counts()
print("Contagem por linguagem:")
print(lang_counts.to_frame("repositórios").to_string())

total_com_lang = df["linguagem"].notna().sum()
em_populares = df["linguagem"].isin(LINGUAGENS_POPULARES).sum()
pct = round(em_populares / total_com_lang * 100, 1)

print(f"\nRepositórios com linguagem definida: {total_com_lang}")
print(f"Repositórios em linguagens populares (Top 3 Octoverse): {em_populares} ({pct}%)")

Contagem por linguagem:
                  repositórios
linguagem                     
Python                     201
TypeScript                 160
JavaScript                 115
N/A                         95
Go                          76
Rust                        54
Java                        47
C++                         46
C                           25
Jupyter Notebook            23
Shell                       21
HTML                        18
Ruby                        12
C#                          11
Kotlin                      10
CSS                          8
PHP                          7
Vue                          7
Swift                        7
Dart                         6
Markdown                     5
MDX                          5
Clojure                      4
Vim Script                   4
Zig                          3
Makefile                     3
Dockerfile                   3
Assembly                     2
Nunjucks                     2
TeX            

## 5. RQ07 (Bônus) — Linguagens populares vs. outras: PRs, releases e frequência de atualização

Comparação das medianas de RQ02, RQ03 e RQ04 entre repositórios escritos nas **linguagens mais populares** (TypeScript, Python, JavaScript — Octoverse 2025) e repositórios em **outras linguagens**.

In [8]:
df_lang = df[df["linguagem"].notna()].copy()
df_lang["grupo"] = df_lang["linguagem"].apply(
    lambda x: "Top 3 Octoverse" if x in LINGUAGENS_POPULARES else "Outras linguagens"
)

metricas = ["prs_aceitas", "releases", "dias_desde_update"]
nomes = {
    "prs_aceitas": "RQ02 - PRs aceitas",
    "releases": "RQ03 - Releases",
    "dias_desde_update": "RQ04 - Dias desde último update",
}

rq07 = df_lang.groupby("grupo")[metricas].median().rename(columns=nomes)
rq07.index.name = "Grupo"

print("Medianas por grupo de linguagem:\n")
print(rq07.to_string())

print(f"\nQuantidade — Top 3 Octoverse: {(df_lang['grupo'] == 'Top 3 Octoverse').sum()}")
print(f"Quantidade — Outras linguagens: {(df_lang['grupo'] == 'Outras linguagens').sum()}")

Medianas por grupo de linguagem:

                   RQ02 - PRs aceitas  RQ03 - Releases  RQ04 - Dias desde último update
Grupo                                                                                  
Outras linguagens               787.0             43.0                              0.0
Top 3 Octoverse                 957.0             63.5                              0.0

Quantidade — Top 3 Octoverse: 476
Quantidade — Outras linguagens: 429
